In [12]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 15, 'lines.linewidth': 2,
    'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 1200,
})

import yaml
import numpy as np

import os
import sys
sys.path.append(f'{os.getcwd()}/irc_gym')

from irc.manager import IRCManager

from auditoryforage.utils import plot_AF_episode

In [13]:
from auditoryforage.AF_env import AuditoryForaging
env = AuditoryForaging()

In [14]:
from read_data import DataToEpisode

In [ ]:
filename = f'../data_flies/compiled/compiled.csv'
data_to_episode = DataToEpisode(filename = filename, env = env)

In [ ]:
# data_IRC = data_to_episode.data_for_IRC()
# from auditoryforage.utils import save_pickle_file
# file_name = './store/exp_data.pkl'
# save_pickle_file(data_IRC, file_name)

## Visualize chosen episode

In [ ]:
data_IRC = data_to_episode.data_for_IRC()

exp_dict = data_IRC['high_reward_blocks']['episodes'][1][0]
episode = {}
episode['states'] = exp_dict['states']
episode['actions'] = exp_dict['actions']
episode['observations'] = exp_dict['observations']
episode['num_steps'] = len(exp_dict['actions'])

# wrong info
episode['rewards'] = exp_dict['received_food'] # does not incorporate other costs
episode['q_probs'] = np.zeros((len(episode['states'])+1,302)) # all set to 0 temporarily

fig = plot_AF_episode(episode = episode, env = env, agent = None)

## Arbitary sanity check

In [ ]:
import numpy as np
lick_actions = np.array(episode['actions'])//2
lick_actions_inds = np.where(lick_actions == 1)
states = episode['states'].flatten()
chosen_state_inds = np.where(states == 151)
print(lick_actions_inds)
print(chosen_state_inds)

## Summary of actual data

In [34]:
from auditoryforage.utils import open_pickle_file
reward_level = 'high'

no_signal_and_noise_nodes = env.no_signal_nodes + 1
file_name = './store/exp_data.pkl'
data_IRC = open_pickle_file(file_name)
episodes_collection = data_IRC[reward_level + '_reward_blocks']['episodes']
no_episodes = len(episodes_collection)
success_streak_list, success_rate_list, episode_time_list = [], [], []
ep_counter = 0
for episode_no in range(no_episodes):
    no_trials = len(episodes_collection[episode_no])
    for trial_no in range(no_trials):
        episode = episodes_collection[episode_no][trial_no]
        episode_states = episode['states']
        success_streak_list.append(len([i for i in range(len(episode_states)) if episode_states[i] < no_signal_and_noise_nodes and episode_states[i-1] >= no_signal_and_noise_nodes]))
        success_rate_list.append(success_streak_list[-1]/len(episode_states))
        episode_time_list.append(len(episode_states))
        ep_counter += 1
average_success_streak = sum(success_streak_list)/ep_counter
average_success_rate = sum(success_rate_list)/ep_counter
average_episode_time = sum(episode_time_list)/ep_counter


In [33]:
# Low
print(episode_time_list)
print(average_success_streak)
print(average_success_rate)
print(average_episode_time)


[2356, 1715, 242, 20, 50, 2844, 61, 9011, 194, 130, 5315, 87, 4032, 650, 9860, 315]
8.0
0.010478844256503507
2305.125


In [35]:
# High
print(episode_time_list)
print(average_success_streak)
print(average_success_rate)
print(average_episode_time)


[28, 31, 535, 14, 72, 29, 105, 125, 127, 112, 97, 112, 945, 57, 103, 285, 29, 80, 305, 140, 73, 108, 18, 66, 160, 695, 2205, 5, 33, 4, 106, 30, 54, 33, 6358, 23, 80]
2.189189189189189
0.03282067263505205
361.6756756756757
